# 03 · Kaggle: Piper Fon voice on JML (multi-speaker, warm-started)

Fine-tunes a multi-speaker **Piper (VITS medium)** voice on the cleanest JML speakers, warm-started from the
English `en_US-ljspeech-medium` checkpoint (public-domain data), with Fon characters and tone marks as symbols.
Same recipe as `02-first_piper.ipynb` (Mac), at full scale, then compares it with **MMS-TTS-Fon** and the real
recordings on held-out sentences.

**Evaluation** (end of training only): each system reads the same held-out sentences, then is scored by
- **kanbior-asr** (WER/CER, with and without tones; trained on JML, so it may favour our voice);
- **MMS ASR** `facebook/mms-1b-all` (CER only: its WER is inflated by word-splitting; Bible data, so it may
  favour MMS-TTS). If our voice wins under both, the result holds.
The real recordings give each ASR's own error floor. The held-out sentences are real recordings by speakers
kanbior-asr never trained on (its val/test speakers, recomputed with its own split script), so its floor is honest too.
The TTS training data is chosen for quality only (cleanest speakers), independently of the ASR split; our voice reads
the held-out sentences in the voices of its cleanest training speakers. The final checkpoint is picked among the last few
by these scores (not by `val_mel`, which stops improving early in VITS). Native listening comes after.

**Privacy:** the trained model holds each volunteer's voice (one speaker ID each). It is pushed to a
**private** repo and must not be published; the released voice will be an average of the speakers (later step).

**Kaggle setup (once)**
1. *Settings → Accelerator*: **GPU T4 x2** (one T4 is used). *Internet*: **on**.
2. *Add-ons → Secrets*: `HF_TOKEN` with **write** access (checkpoints go to a private repo on your account;
   it also reads the private `kanbior-fon-asr-v1`).
3. Commit and push `kanbior-tts/train_piper.py` and `prepare_data.py` to `REPO_BRANCH` (the notebook clones it).
4. Run with **Save Version → Save & Run All (Commit)**. Each session trains until `TIME_BUDGET_HOURS`, pushes
   `last.ckpt`, and the next session resumes from it. The evaluation runs in the session that reaches `MAX_EPOCHS`.

**Not measured yet:** Piper's speed on a T4. Session 1 prints the time per epoch and a projection: adjust
`MAX_EPOCHS` before session 2 if needed (the learning-rate schedule is spread over `MAX_EPOCHS`).

In [ ]:
import os
import time
SESSION_START = time.time()
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one T4

from pathlib import Path

# ---- Configuration ----
HUB_REPO_NAME = "kanbior-fon-tts-v1"     # private repo on your HF account: data lists, checkpoints, final voice
TARGET_HOURS = 10.0          # training audio: the cleanest speakers, added until this many hours
EVAL_CLIPS = 200             # held-out sentences: real recordings by speakers kanbior-asr never trained on,
                             # texts removed from TTS training
VOICE_SPEAKERS = 10          # our voice reads them in the voices of its 10 cleanest training speakers (round-robin)
# kanbior-asr's split (clips, speakers) from its training run: checked to make sure it is reproduced exactly
ASR_SPLIT = {"train": (50_675, 390), "val": (6_837, None), "test": (6_724, 137)}
MAX_EPOCHS = 150             # total across sessions (to adjust after session 1: see its projection)
CKPT_EVERY = 10              # keep a checkpoint every N epochs ...
CKPT_KEEP = 3                # ... the last 3 are the candidates for the final model (+ last.ckpt)
BATCH_SIZE = 16              # T4 16 GB (Piper's docs use 32 on 24 GB cards)
TIME_BUDGET_HOURS = 10.0     # from session start: training stops here (Kaggle limit 12 h)
PUSH_EVERY_MIN = 30          # push new checkpoints to the Hub this often during training

REPO_URL = "https://github.com/dyeto/kanbior.git"
REPO_BRANCH = "feat/kanbior-tts"
PIPER_URL = "https://github.com/OHF-Voice/piper1-gpl.git"
PIPER_COMMIT = "6aacdbc"     # piper1-gpl version the recipe was checked against
JML_REPO = "JaimeMaLangue/JML-fon-speech-v1"
ARCHIVES = range(8)          # all transcribed audio (~15.5 GB); speakers are picked from all of it
BASE_REPO, BASE_DIR = "rhasspy/piper-checkpoints", "en/en_US/ljspeech/medium"
KANBIOR_ASR_NAME = "kanbior-fon-asr-v1"  # on your HF account
MMS_TTS = "facebook/mms-tts-fon"         # baseline voice (CC-BY-NC: comparison only)
MMS_ASR = "facebook/mms-1b-all"          # second scorer (CC-BY-NC: evaluation only)

SAMPLE_RATE = 22_050
NUM_SYMBOLS = 256
WORK = Path("/kaggle/working")           # small outputs (log, results)
DATA_DIR = Path("/tmp/jml")              # JML audio
SPLITS_DIR = Path("/tmp/tts-splits")     # prepare_data.py output
RUN_DIR = Path("/tmp/piper-run")         # Piper cache and checkpoints (large, not kept in the notebook output)
VOICE_NAME = "kanbior-fon-tts-v1"

In [ ]:
import shutil
import subprocess
import sys

# Piper training code (GPL-3.0; the trained voice is not covered by it, see docs/LICENSING.md)
PIPER_DIR = Path("/tmp/piper1-gpl")
if not PIPER_DIR.exists():
    subprocess.run(["git", "clone", "-q", PIPER_URL, str(PIPER_DIR)], check=True)
    subprocess.run(["git", "-C", str(PIPER_DIR), "checkout", "-q", PIPER_COMMIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{PIPER_DIR}[train]", "jiwer>=4.0", "transformers>=5.24,<6"], check=True)
    subprocess.run(["bash", "build_monotonic_align.sh"], cwd=PIPER_DIR, check=True)

# The editable install registers Piper through a .pth file, read only at Python startup: add it to this kernel too.
import importlib
import site
for d in site.getsitepackages():
    site.addsitedir(d)
importlib.invalidate_caches()
from piper import PiperVoice, SynthesisConfig   # noqa: E402  (checks the install)
print("piper", subprocess.run(["git", "-C", str(PIPER_DIR), "log", "-1", "--format=%h %cd"],
                              capture_output=True, text=True).stdout.strip())

In [ ]:
import csv
import json
import random
import re
import unicodedata
from collections import Counter

import jiwer
import librosa
import numpy as np
import soundfile as sf
import torch
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import Audio, display

# kanbior code: TTS data prep, Piper launcher, Fon text normalization
CODE_DIR = Path("/tmp/kanbior")
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(CODE_DIR)], check=True)
sys.path.insert(0, str(CODE_DIR / "kanbior-mcore"))
from fon_text import clean_text, normalize_for_wer   # noqa: E402
assert (CODE_DIR / "kanbior-tts" / "train_piper.py").is_file(), f"push train_piper.py to {REPO_BRANCH}"

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi()
USER = api.whoami()["name"]
HUB_ID = f"{USER}/{HUB_REPO_NAME}"
api.create_repo(HUB_ID, private=True, exist_ok=True)
hub_files = set(api.list_repo_files(HUB_ID))
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("Hub repo: https://huggingface.co/" + HUB_ID, f"({len(hub_files)} files)")

## 1. Data

Downloads all transcribed JML audio, then `kanbior-tts/prepare_data.py` measures every clip and drops noisy,
clipped, mostly silent or mismatched ones. The first session picks the speakers and the held-out sentences
and pushes the lists to the Hub (`data/`); later sessions reuse them, so every session trains on the same data.

In [ ]:
def fetch_archive(i):
    """Download one JML archive, extract it into DATA_DIR, delete the tar (keeps disk use low)."""
    name = f"audio_transcribed_{i:03d}.tar"
    marker = DATA_DIR / f".{name}.done"
    if marker.exists():
        return
    free_gb = shutil.disk_usage(DATA_DIR).free / 1e9
    assert free_gb > 5, f"Only {free_gb:.1f} GB free; need ~5 GB per archive"
    t = time.time()
    tar = hf_hub_download(JML_REPO, name, repo_type="dataset", local_dir=DATA_DIR / "_download")
    subprocess.run(["tar", "-xf", tar, "-C", str(DATA_DIR)], check=True)
    os.remove(tar)
    marker.touch()
    print(f"{name}: {time.time() - t:.0f} s, {shutil.disk_usage(DATA_DIR).free / 1e9:.0f} GB free")

def run_logged(cmd, **kw):
    """Run a command, printing its output as it comes (so long steps show progress, also in commit logs)."""
    t = time.time()
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **kw) as p:
        for line in p.stdout:
            print(line, end="", flush=True)
    assert p.returncode == 0, f"failed ({p.returncode}): {' '.join(map(str, cmd[:3]))}"
    print(f"  [{(time.time() - t) / 60:.1f} min]", flush=True)

def hub_get(path, local_dir):
    return Path(hf_hub_download(HUB_ID, path, local_dir=local_dir))

def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def write_csv(path, rows, fields=("id", "audio_path", "speaker_id", "text", "duration")):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(fields), extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)

def tts_text(text):   # what the voice reads: lowercase (no capitals in the symbol map), no "|" (CSV separator)
    return unicodedata.normalize("NFC", text.lower().replace("|", " ")).strip()

DATA_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
hf_hub_download(JML_REPO, "transcribed.csv", repo_type="dataset", local_dir=DATA_DIR)
for i in ARCHIVES:
    fetch_archive(i)

In [ ]:
base_ckpt = hf_hub_download(BASE_REPO, f"{BASE_DIR}/lj-med_1000.ckpt", repo_type="dataset")   # ~846 MB
base_config = json.loads(Path(hf_hub_download(BASE_REPO, f"{BASE_DIR}/config.json", repo_type="dataset")).read_text())

LISTS = ["train.csv", "eval.csv", "fon_phonemes.json"]
if all(f"data/{f}" in hub_files for f in LISTS):
    for f in LISTS:
        hub_get(f"data/{f}", RUN_DIR.parent / "hub-data")
    for f in LISTS:
        shutil.copy(RUN_DIR.parent / "hub-data" / "data" / f, RUN_DIR / f)
    print("Reusing the data lists of session 1 from the Hub")
else:
    if "data/audio_stats.csv" in hub_files:   # skips re-measuring 64k clips if a session stopped here
        shutil.copy(hub_get("data/audio_stats.csv", RUN_DIR.parent / "hub-data"), SPLITS_DIR / "audio_stats.csv")
    print("Measuring all clips (prints every 1,000; ~15-30 min)...", flush=True)
    run_logged([sys.executable, "-u", str(CODE_DIR / "kanbior-tts" / "prepare_data.py"),
                "--data-dir", str(DATA_DIR), "--out-dir", str(SPLITS_DIR)])
    api.upload_file(path_or_fileobj=SPLITS_DIR / "audio_stats.csv", path_in_repo="data/audio_stats.csv", repo_id=HUB_ID)

    # kanbior-asr's speaker split, recomputed with its own script on the same audio (all 8 archives),
    # as in kanbior-asr/02. Its val and test speakers were never trained on (val only reported scores).
    asr_dir = SPLITS_DIR / "asr"
    run_logged([sys.executable, "-u", str(CODE_DIR / "kanbior-asr" / "prepare_data.py"),
                "--data-dir", str(DATA_DIR), "--out-dir", str(asr_dir)])
    asr = {name: read_csv(asr_dir / f"{name}.csv") for name in ("train", "val", "test")}
    got = {name: (len(rows), len({r["speaker_id"] for r in rows})) for name, rows in asr.items()}
    assert (got["train"] == ASR_SPLIT["train"] and got["test"] == ASR_SPLIT["test"]
            and got["val"][0] == ASR_SPLIT["val"][0]), f"kanbior-asr split not reproduced: {got} vs {ASR_SPLIT}"
    asr_train_speakers = {r["speaker_id"] for r in asr["train"]}
    asr_unseen_speakers = {r["speaker_id"] for name in ("val", "test") for r in asr[name]}
    asr_train_texts = {normalize_for_wer(r["text"]) for r in asr["train"]}

    # TTS speakers: chosen for quality only, cleanest first (speakers.csv is sorted by clean hours) until TARGET_HOURS
    pool_all = read_csv(SPLITS_DIR / "train.csv") + read_csv(SPLITS_DIR / "val.csv")
    hours_by_spk = Counter()
    for r in pool_all:
        hours_by_spk[r["speaker_id"]] += float(r["duration"]) / 3600
    ranked = [r["speaker_id"] for r in read_csv(SPLITS_DIR / "speakers.csv") if hours_by_spk[r["speaker_id"]] > 0]
    chosen_ranked, total = [], 0.0
    for spk in ranked:
        if total >= TARGET_HOURS:
            break
        chosen_ranked.append(spk)
        total += hours_by_spk[spk]
    chosen = set(chosen_ranked)

    # Held-out sentences: real recordings from kanbior-asr's val/test speakers (never trained on), with the
    # same audio quality filters as the TTS data, spread over speakers, distinct texts, texts kanbior-asr never
    # trained on first. Each is read by our voice in one of its VOICE_SPEAKERS cleanest training speakers.
    stats = {r["id"]: r for r in read_csv(SPLITS_DIR / "audio_stats.csv")}
    def clean_clip(r):
        m = stats.get(r["id"])
        return bool(m) and 1.0 <= float(m["duration"]) <= 15.0 and float(m["snr_db"]) >= 25.0 \
            and float(m["clip_frac"]) <= 0.001 and float(m["speech_frac"]) >= 0.4
    by_spk = {}
    for r in sorted((r for name in ("val", "test") for r in asr[name]), key=lambda r: r["id"]):
        if clean_clip(r):
            by_spk.setdefault(r["speaker_id"], []).append(r)
    rng = random.Random(42)
    eval_speakers = sorted(by_spk)
    rng.shuffle(eval_speakers)
    for rows in by_spk.values():
        rng.shuffle(rows)
        rows.sort(key=lambda r: normalize_for_wer(r["text"]) in asr_train_texts)   # stable: unseen texts first
    voices_for_eval = chosen_ranked[:VOICE_SPEAKERS]
    eval_rows, eval_texts = [], set()
    while len(eval_rows) < EVAL_CLIPS and any(by_spk.values()):
        for spk in eval_speakers:   # one clip per speaker per round
            while by_spk[spk] and len(eval_rows) < EVAL_CLIPS:
                r = by_spk[spk].pop(0)
                if (t := normalize_for_wer(r["text"])) not in eval_texts:
                    eval_rows.append({**r, "voice": voices_for_eval[len(eval_rows) % len(voices_for_eval)]})
                    eval_texts.add(t)
                    break
    assert len(eval_rows) == EVAL_CLIPS, f"only {len(eval_rows)} clean held-out clips"
    assert not {r["speaker_id"] for r in eval_rows} & asr_train_speakers

    pool = sorted((r for r in pool_all if r["speaker_id"] in chosen), key=lambda r: r["id"])
    train_rows = [r for r in pool if normalize_for_wer(r["text"]) not in eval_texts]
    unseen_text = sum(t not in asr_train_texts for t in eval_texts)
    per_spk = Counter(r["speaker_id"] for r in eval_rows)
    print(f"{len(chosen)} TTS speakers | train {len(train_rows):,} clips "
          f"({sum(float(r['duration']) for r in train_rows) / 3600:.1f} h; {len(pool) - len(train_rows)} dropped: "
          f"same text as a held-out sentence)")
    print(f"eval {len(eval_rows)} clips from {len(per_spk)} speakers unseen by kanbior-asr "
          f"({min(per_spk.values())}-{max(per_spk.values())} each); {unseen_text} of their texts "
          f"({100 * unseen_text / len(eval_texts):.0f}%) never seen by kanbior-asr either | "
          f"read by our voice as {len(voices_for_eval)} speakers")
    write_csv(RUN_DIR / "train.csv", train_rows)
    write_csv(RUN_DIR / "eval.csv", eval_rows, fields=("id", "audio_path", "speaker_id", "text", "duration", "voice"))

    # Fon symbol map (as in 02-first_piper): English ids for shared symbols, g -> IPA ɡ, free ids for the rest
    fon_map = {k: list(v) for k, v in base_config["phoneme_id_map"].items()}
    used = {i for ids in fon_map.values() for i in ids}
    free_ids = (i for i in range(NUM_SYMBOLS) if i not in used)
    fon_map["g"] = fon_map["ɡ"]
    for ch in sorted({ch for r in train_rows + eval_rows for ch in unicodedata.normalize("NFD", tts_text(r["text"]))}):
        if ch not in fon_map:
            fon_map[ch] = [next(free_ids)]
            print(f"  new symbol U+{ord(ch):04X} {unicodedata.name(ch, '?')} -> id {fon_map[ch][0]}")
    (RUN_DIR / "fon_phonemes.json").write_text(json.dumps(fon_map, ensure_ascii=False, indent=1), encoding="utf-8")

    for f in LISTS:
        api.upload_file(path_or_fileobj=RUN_DIR / f, path_in_repo=f"data/{f}", repo_id=HUB_ID)

train_rows, eval_rows = read_csv(RUN_DIR / "train.csv"), read_csv(RUN_DIR / "eval.csv")
fon_map = json.loads((RUN_DIR / "fon_phonemes.json").read_text(encoding="utf-8"))
speakers = sorted({r["speaker_id"] for r in train_rows})

uncovered = Counter(ch for r in train_rows + eval_rows for ch in unicodedata.normalize("NFD", tts_text(r["text"]))
                    if ch not in fon_map)
assert not uncovered, f"Symbols missing from the map: {uncovered}"
assert not [r for r in train_rows + eval_rows if not (DATA_DIR / r["audio_path"]).is_file()], "audio missing"

with open(RUN_DIR / "metadata.csv", "w", encoding="utf-8") as f:   # wav|speaker|text, no header
    for r in train_rows:
        f.write(f"{r['audio_path']}|{r['speaker_id']}|{tts_text(r['text'])}\n")

# Free disk: keep only the audio we use (Piper's cache adds ~1 GB per training hour)
keep = {r["audio_path"] for r in train_rows + eval_rows}
removed = 0
for p in (DATA_DIR / "audio" / "transcribed").glob("*.wav"):
    if f"audio/transcribed/{p.name}" not in keep:
        p.unlink()
        removed += 1
print(f"{len(speakers)} speakers, train {len(train_rows):,} clips "
      f"({sum(float(r['duration']) for r in train_rows) / 3600:.1f} h), eval {len(eval_rows)} | "
      f"removed {removed:,} unused wavs, {shutil.disk_usage(RUN_DIR).free / 1e9:.0f} GB free")

## 2. Train (resumes from the last pushed checkpoint)

Runs `kanbior-tts/train_piper.py` (Piper's trainer with our checkpoints: one every `CKPT_EVERY` epochs, the
last `CKPT_KEEP` kept, plus `last.ckpt`). A new run warm-starts from the English voice (`--model.warmstart_ckpt`);
later sessions resume with `--ckpt_path` (epoch counter, optimizer and settings restored).
Checkpoints are pushed to the Hub every `PUSH_EVERY_MIN` minutes and at the end of the session.

In [ ]:
def ckpt_epoch(path):
    return torch.load(path, map_location="cpu", weights_only=False, mmap=True)["epoch"]

def local_ckpts():
    return sorted((RUN_DIR / "lightning_logs").glob("version_*/checkpoints/*.ckpt"), key=lambda p: p.stat().st_mtime)

pushed = {}   # file name -> mtime already pushed
def push_checkpoints(final=False):
    """Push last.ckpt and new periodic checkpoints; keep only the newest CKPT_KEEP periodic ones on the Hub."""
    stage = RUN_DIR / "stage"
    stage.mkdir(exist_ok=True)
    latest = {}
    for p in local_ckpts():   # same name in several version_* dirs: the newest wins
        latest[p.name] = p
    for name, p in latest.items():
        mtime = p.stat().st_mtime
        if pushed.get(name) == mtime or (not final and time.time() - mtime < 30):   # skip files being written
            continue
        t = time.time()
        shutil.copy(p, stage / name)
        api.upload_file(path_or_fileobj=stage / name, path_in_repo=f"checkpoints/{name}", repo_id=HUB_ID,
                        commit_message=f"checkpoint {name}")
        (stage / name).unlink()
        pushed[name] = mtime
        print(f"  [hub] pushed {name} (epoch {ckpt_epoch(p) + 1}) in {time.time() - t:.0f} s", flush=True)
    if (RUN_DIR / "config.json").is_file():
        api.upload_file(path_or_fileobj=RUN_DIR / "config.json", path_in_repo="data/config.json", repo_id=HUB_ID)
    periodic = sorted(f for f in api.list_repo_files(HUB_ID) if re.fullmatch(r"checkpoints/epoch=\d+\.ckpt", f))
    for old in periodic[:-CKPT_KEEP]:
        api.delete_file(old, repo_id=HUB_ID, commit_message=f"drop old {old}")

resume = None
start_epoch = 0
if "checkpoints/last.ckpt" in hub_files:
    resume = hub_get("checkpoints/last.ckpt", RUN_DIR / "resume")
    start_epoch = ckpt_epoch(resume) + 1
    print(f"Resuming after epoch {start_epoch} of {MAX_EPOCHS}")
else:
    print("Starting a new run (warm start from the English voice)")
FINISHED = start_epoch >= MAX_EPOCHS

In [ ]:
if not FINISHED:
    remaining_min = int((TIME_BUDGET_HOURS * 3600 - (time.time() - SESSION_START)) / 60)
    assert remaining_min > 30, "not enough session time left to train"
    cmd = [
        sys.executable, str(CODE_DIR / "kanbior-tts" / "train_piper.py"), "fit",
        "--data.voice_name", VOICE_NAME,
        "--data.csv_path", str(RUN_DIR / "metadata.csv"),
        "--data.audio_dir", str(DATA_DIR),
        "--data.cache_dir", str(RUN_DIR / "cache"),
        "--data.config_path", str(RUN_DIR / "config.json"),
        "--data.espeak_voice", "en",               # required by the CLI, unused in text mode
        "--data.phoneme_type", "text",
        "--data.phonemes_path", str(RUN_DIR / "fon_phonemes.json"),
        "--data.num_symbols", str(NUM_SYMBOLS),
        "--data.batch_size", str(BATCH_SIZE),
        "--data.num_workers", "4",
        "--data.length_bucketing", "true",         # less padding; set for the whole run
        "--model.num_speakers", str(len(speakers)),
        "--model.sample_rate", str(SAMPLE_RATE),
        "--model.mos_metric", "none",
        "--trainer.accelerator", "gpu",
        "--trainer.devices", "1",
        "--trainer.max_epochs", str(MAX_EPOCHS),
        "--trainer.max_time", f"00:{remaining_min // 60:02d}:{remaining_min % 60:02d}:00",
        "--trainer.log_every_n_steps", "50",
        "--trainer.default_root_dir", str(RUN_DIR),
    ]
    cmd += ["--ckpt_path", str(resume)] if resume else ["--model.warmstart_ckpt", base_ckpt]
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PIPER_CKPT_EVERY": str(CKPT_EVERY), "PIPER_CKPT_KEEP": str(CKPT_KEEP)}
    print(f"Training for at most {remaining_min} min")

    t0 = last_push = last_beat = time.time()
    log_lines, epoch_seen, last_bar = [], None, ""
    bar = re.compile(r"\|\s*\d+/\d+ \[")   # tqdm progress bar: "...| 9/20 [00:05<..."
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as proc:
        for line in proc.stdout:
            log_lines.append(line)
            if not bar.search(line):   # progress-bar redraws are summarized below, not printed
                print(line, end="", flush=True)
            else:
                last_bar = line.strip()
                m = re.search(r"Epoch (\d+):", line)
                if m and m.group(1) != epoch_seen:   # one line when an epoch starts
                    epoch_seen = m.group(1)
                    print(f"[{(time.time() - t0) / 60:6.1f} min] epoch {int(epoch_seen) + 1}/{MAX_EPOCHS}", flush=True)
            if time.time() - last_beat > 10 * 60:   # heartbeat: latest progress bar, every 10 min
                print(f"[{(time.time() - t0) / 60:6.1f} min] {last_bar[-160:]}", flush=True)
                last_beat = time.time()
            if time.time() - last_push > PUSH_EVERY_MIN * 60:
                push_checkpoints()
                last_push = time.time()
    log = "".join(log_lines)
    (WORK / f"train-from-epoch-{start_epoch}.log").write_text(log, encoding="utf-8")
    print(f"\nExit code {proc.returncode}, {(time.time() - t0) / 3600:.2f} h")
    assert proc.returncode == 0, "training failed: see the log above"
    assert "Missing phoneme from id map" not in log, "some symbols were dropped: check the map"
    if not resume:
        assert "[warmstart] Copied" in log, "warm-start did not run"

    push_checkpoints(final=True)
    last = [p for p in local_ckpts() if p.name == "last.ckpt"][-1]
    end_epoch = ckpt_epoch(last) + 1
    FINISHED = end_epoch >= MAX_EPOCHS
    done = end_epoch - start_epoch
    if done:
        per_epoch = (time.time() - t0) / done   # includes preprocessing on the first session: an upper bound
        print(f"Epochs {start_epoch} -> {end_epoch} of {MAX_EPOCHS}: ~{per_epoch / 60:.1f} min per epoch; "
              f"the rest needs ~{(MAX_EPOCHS - end_epoch) * per_epoch / 3600:.1f} h "
              f"(~{(MAX_EPOCHS - end_epoch) * per_epoch / 3600 / TIME_BUDGET_HOURS:.1f} more sessions)")
print("FINISHED" if FINISHED else "Not finished: run the notebook again (Save & Run All) to continue")

## 3. Evaluation (only when training is finished)

Candidates: the last `CKPT_KEEP` periodic checkpoints (+ `last.ckpt` if it is newer), each exported to ONNX.
Every system reads the `EVAL_CLIPS` held-out sentences: our candidates in the voice assigned to each sentence
(`voice` column: one of the cleanest training speakers),
MMS-TTS with its single voice. The real recordings of the same sentences are the floor.

In [ ]:
if FINISHED:
    EVAL_DIR = Path("/tmp/eval")
    cand_dir = RUN_DIR / "candidates"
    names = [f for f in api.list_repo_files(HUB_ID) if f.startswith("checkpoints/") and f.endswith(".ckpt")]
    ckpts = {}
    for f in names:
        p = hub_get(f, cand_dir)
        ckpts[f"epoch {ckpt_epoch(p) + 1}"] = p   # last.ckpt and the final periodic one can be the same epoch
    if not (RUN_DIR / "config.json").is_file():
        shutil.copy(hub_get("data/config.json", cand_dir), RUN_DIR / "config.json")

    voices = {}
    for label, ckpt in sorted(ckpts.items(), key=lambda kv: int(kv[0].split()[1])):
        onnx = RUN_DIR / "onnx" / f"{label.replace(' ', '-')}.onnx"
        subprocess.run([sys.executable, "-m", "piper.train.export_onnx", "--checkpoint", str(ckpt),
                        "--output-file", str(onnx)], check=True)
        shutil.copy(RUN_DIR / "config.json", f"{onnx}.json")
        voices[f"kanbior {label}"] = onnx
        print(f"exported {label}", flush=True)
    print("Candidates:", ", ".join(voices), f"| ONNX {next(iter(voices.values())).stat().st_size / 1e6:.0f} MB each")

    texts = {r["id"]: tts_text(r["text"]) for r in eval_rows}
    audio = {"real recordings": {r["id"]: DATA_DIR / r["audio_path"] for r in eval_rows}}
    rtf = {}
    for system, onnx in voices.items():   # CPU, as in production
        voice = PiperVoice.load(onnx, config_path=f"{onnx}.json")
        out_dir = EVAL_DIR / system.replace(" ", "_")
        out_dir.mkdir(parents=True, exist_ok=True)
        audio[system], gen_s, audio_s = {}, 0.0, 0.0
        for n, r in enumerate(eval_rows, 1):
            if n % 50 == 0:
                print(f"  {system}: {n}/{len(eval_rows)} sentences", flush=True)
            t = time.time()
            chunks = list(voice.synthesize(texts[r["id"]],
                                           SynthesisConfig(speaker_id=voice.config.speaker_id_map[r["voice"]])))
            gen_s += time.time() - t
            wav = np.concatenate([c.audio_float_array for c in chunks])
            audio_s += len(wav) / chunks[0].sample_rate
            sf.write(out_dir / f"{r['id']}.wav", wav, chunks[0].sample_rate)
            audio[system][r["id"]] = out_dir / f"{r['id']}.wav"
        rtf[system] = gen_s / audio_s
        print(f"{system}: real-time factor {rtf[system]:.3f} on CPU")

In [ ]:
if FINISHED:
    from transformers import AutoTokenizer, VitsModel

    mms_tts = VitsModel.from_pretrained(MMS_TTS).to("cuda").eval()
    mms_tok = AutoTokenizer.from_pretrained(MMS_TTS)
    out_dir = EVAL_DIR / "mms-tts"
    out_dir.mkdir(parents=True, exist_ok=True)
    audio["MMS-TTS-Fon"] = {}
    torch.manual_seed(0)   # MMS-TTS samples durations: fixed seed, reproducible scores
    for r in eval_rows:
        inputs = mms_tok(texts[r["id"]], return_tensors="pt").to("cuda")
        with torch.no_grad():
            wav = mms_tts(**inputs).waveform[0].cpu().numpy()
        sf.write(out_dir / f"{r['id']}.wav", wav, mms_tts.config.sampling_rate)
        audio["MMS-TTS-Fon"][r["id"]] = out_dir / f"{r['id']}.wav"
    del mms_tts
    torch.cuda.empty_cache()
    print("MMS-TTS-Fon:", len(audio["MMS-TTS-Fon"]), "sentences")

### Scoring with two ASR models

The real recordings come from speakers kanbior-asr never trained on, so both floors are honest. The floor and
our voice are different speakers (our voice reads in its training voices): read each system's error as
"above the ASR's own level on human speech". kanbior-asr knows our voice's training speakers, which may favour
it; MMS ASR never saw JML and balances that. kanbior-asr may also have seen some *texts* read by other speakers
(JML repeats sentences): section 1 prints how many are new to it.


In [ ]:
def score(refs, preds):
    out = {}
    for suffix, keep in (("", True), ("_no_tones", False)):
        r = [normalize_for_wer(t, keep) for t in refs]
        p = [normalize_for_wer(t, keep) for t in preds]
        out["wer" + suffix] = 100 * jiwer.wer(r, p)
        out["cer" + suffix] = 100 * jiwer.cer(r, p)
    return out

def load_16k(path):
    wav, sr = sf.read(path, dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)
    return librosa.resample(wav, orig_sr=sr, target_sr=16_000) if sr != 16_000 else wav

if FINISHED:
    from transformers import AutoProcessor, Wav2Vec2ForCTC, WhisperForConditionalGeneration, WhisperProcessor

    ids = [r["id"] for r in eval_rows]
    refs = [r["text"] for r in eval_rows]
    transcripts = {"kanbior-asr": {}, "MMS ASR": {}}

    # kanbior-asr (whisper-small fine-tuned on JML), fp16 on the T4
    asr_id = f"{USER}/{KANBIOR_ASR_NAME}"
    proc = WhisperProcessor.from_pretrained(asr_id, language="french", task="transcribe")
    asr = WhisperForConditionalGeneration.from_pretrained(asr_id, dtype=torch.float16).to("cuda").eval()
    asr.generation_config.language, asr.generation_config.task = "fr", "transcribe"
    asr.generation_config.forced_decoder_ids = None
    for system, files in audio.items():
        t = time.time()
        preds = []
        for i in range(0, len(ids), 16):
            feats = proc.feature_extractor([load_16k(files[k]) for k in ids[i:i + 16]], sampling_rate=16_000,
                                           return_tensors="pt").input_features.to("cuda", torch.float16)
            with torch.no_grad():
                out = asr.generate(feats, max_length=225)
            preds += [t.strip() for t in proc.tokenizer.batch_decode(out, skip_special_tokens=True)]
        transcripts["kanbior-asr"][system] = preds
        print(f"kanbior-asr | {system}: {time.time() - t:.0f} s", flush=True)
    del asr
    torch.cuda.empty_cache()

    # MMS ASR with the Fon adapter
    mproc = AutoProcessor.from_pretrained(MMS_ASR, target_lang="fon")
    masr = Wav2Vec2ForCTC.from_pretrained(MMS_ASR, target_lang="fon", ignore_mismatched_sizes=True,
                                          dtype=torch.float16).to("cuda").eval()
    for system, files in audio.items():
        t = time.time()
        preds = []
        for k in ids:
            x = mproc(load_16k(files[k]), sampling_rate=16_000, return_tensors="pt").input_values.to("cuda", torch.float16)
            with torch.no_grad():
                preds.append(mproc.decode(masr(x).logits[0].argmax(dim=-1)))
        transcripts["MMS ASR"][system] = preds
        print(f"MMS ASR | {system}: {time.time() - t:.0f} s", flush=True)
    del masr
    torch.cuda.empty_cache()

    results = {s: {"kanbior-asr": score(refs, transcripts["kanbior-asr"][s]),
                   "MMS ASR": score(refs, transcripts["MMS ASR"][s])} for s in audio}
    floor = results["real recordings"]
    print(f"{'':22s} {'kanbior-asr':^33s} | {'MMS ASR':^15s}")
    print(f"{'system':22s} {'WER':>7s} {'no tn':>7s} {'CER':>7s} {'no tn':>7s} | {'CER':>7s} {'no tn':>7s} | CER above floor")
    for s, m in results.items():
        k, mm = m["kanbior-asr"], m["MMS ASR"]
        gap = "" if s == "real recordings" else (f"{k['cer'] - floor['kanbior-asr']['cer']:+.1f} / "
                                                  f"{mm['cer'] - floor['MMS ASR']['cer']:+.1f}")
        print(f"{s:22s} {k['wer']:7.1f} {k['wer_no_tones']:7.1f} {k['cer']:7.1f} {k['cer_no_tones']:7.1f} | "
              f"{mm['cer']:7.1f} {mm['cer_no_tones']:7.1f} | {gap}")
    print("\nFor reference, kanbior-asr on its own test set (unseen speakers): WER 20.5 / 15.5 no tones, CER 8.5 / 6.6")

### Pick the final checkpoint and push it

Lowest CER under both ASR models (sum), among our candidates. The Hub repo stays **private**: this model
holds the volunteers' individual voices.

In [ ]:
if FINISHED:
    ours = [s for s in voices]
    best = min(ours, key=lambda s: results[s]["kanbior-asr"]["cer"] + results[s]["MMS ASR"]["cer"])
    k, b = results[best], results["MMS-TTS-Fon"]
    print(f"Final: {best}")
    print(f"  vs MMS-TTS-Fon, kanbior-asr CER: {k['kanbior-asr']['cer']:.1f} vs {b['kanbior-asr']['cer']:.1f} | "
          f"MMS ASR CER: {k['MMS ASR']['cer']:.1f} vs {b['MMS ASR']['cer']:.1f}")
    both = (k["kanbior-asr"]["cer"] < b["kanbior-asr"]["cer"]) and (k["MMS ASR"]["cer"] < b["MMS ASR"]["cer"])
    print("  ->", "better than MMS-TTS under both ASR models" if both else "not better under both: listening decides")

    final_onnx = WORK / f"{VOICE_NAME}-medium.onnx"
    shutil.copy(voices[best], final_onnx)
    shutil.copy(f"{voices[best]}.json", f"{final_onnx}.json")
    summary = {"final": best, "max_epochs": MAX_EPOCHS, "speakers": len(speakers),
               "train_hours": sum(float(r["duration"]) for r in train_rows) / 3600, "eval_clips": len(eval_rows),
               "results": results, "rtf_cpu": rtf, "onnx_mb": final_onnx.stat().st_size / 1e6}
    (WORK / "results.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    for f in (final_onnx, Path(f"{final_onnx}.json"), WORK / "results.json"):
        api.upload_file(path_or_fileobj=f, path_in_repo=f"final/{f.name}", repo_id=HUB_ID,
                        commit_message=f"Final voice ({best}), CER {k['kanbior-asr']['cer']:.1f}")
    print("Pushed to https://huggingface.co/" + HUB_ID + "/tree/main/final")

In [ ]:
# Listen: real recording, MMS-TTS-Fon, our final voice (same sentence)
if FINISHED:
    for r in eval_rows[:5]:
        print(f"\n{r['id']} (real: {r['speaker_id']}, our voice: {r['voice']}): {r['text']}")
        for s in ("real recordings", "MMS-TTS-Fon", best):
            i = ids.index(r["id"])
            print(f"  {s}  | kanbior-asr: {transcripts['kanbior-asr'][s][i]}")
            display(Audio(str(audio[s][r["id"]])))